# Tenacious-Bench — Day 0 Smoke Test (Colab T4)

**Week 11 · Day 0 pre-flight**

Per the challenge brief: _the Unsloth notebook completes a 5-task dummy LoRA run end to end (fp16 on T4, bf16 on L4/4090) and pushes the adapter to your HuggingFace account._

This notebook proves the training stack works **before Day 5**. If anything in this notebook fails, fix it now — Day 5 has no time to debug compute.

**What it does:**

1. Confirm a T4 (or better) GPU is attached.
2. Install Unsloth + TRL + PEFT.
3. Authenticate to HuggingFace (Colab Secrets → `HF_TOKEN`).
4. Load `unsloth/Qwen3.5-1.7B-Instruct` in 16-bit, attach LoRA.
5. Run **1 SimPO step** on 5 dummy preference pairs.
6. Push the adapter to `<your-hf-user>/tenacious-smoke-test` (private).

**Expected wall time:** 8–15 min (most of which is the first-run kernel compile).

> **QLoRA 4-bit is NOT used** — Week 11 brief mandates 16-bit LoRA.


## Step 1 — Check GPU runtime

Confirm a T4 (or better) is attached. **Runtime → Change runtime type → T4 GPU** if not.


In [ ]:
import subprocess

print(
    subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    or "NO GPU — change runtime to T4"
)
import torch

print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(
        f"bf16 supported: {torch.cuda.is_bf16_supported()}  (T4 → False, fp16; L4/4090 → True, bf16)"
    )

## Step 2 — Clone the repo

Skip this cell if you uploaded the repo manually. Otherwise replace `<your-fork>` with your GitHub fork URL.


In [1]:
import os
REPO_URL = "https://github.com/sanoy24/tenacious-bench.git"  # ← edit me
if not os.path.isdir("/content/tenacious-bench"):
    !git clone $REPO_URL /content/tenacious-bench
%cd /content/tenacious-bench
!ls training/

/content/tenacious-bench


## Step 3 — Install dependencies

First-run kernel compile takes 6–10 min on T4 — that is expected.


In [2]:
# Unsloth first — it pins compatible torch/transformers versions
!pip install -q unsloth
!pip install -q "trl>=0.9.0" "peft>=0.12.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
               sentencepiece protobuf "huggingface_hub>=0.24.0"
import trl, peft, transformers
print(f"trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")

trl=0.24.0  peft=0.19.1  transformers=5.5.0


## Step 4 — HuggingFace authentication

**Recommended:** Colab sidebar → 🔑 lock icon (Secrets) → add a secret named `HF_TOKEN` with your **write-scope** token from <https://huggingface.co/settings/tokens>. Toggle _Notebook access_ on.

The smoke-test script picks the token up automatically (env var → Colab Secrets → interactive fallback).


In [4]:
import os
from huggingface_hub import login

login(os.getenv("HF_TOKEN"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## Step 5 — Run the smoke test

Runs `training/colab_smoke_test.py`:

- Loads `unsloth/Qwen3.5-1.7B-Instruct` in 16-bit (no 4-bit quantization).
- Attaches LoRA (rank=16, α=32) on `q/k/v/o/gate/up/down_proj`.
- Trains **1 SimPO step** on 5 dummy preference pairs.
- Saves adapter and pushes to `<HF_USER>/tenacious-smoke-test` (private).

**Replace `<your-hf-user>` below.** Drop `--hf-repo` to skip the push.


In [5]:
HF_USER = "sanoy24"  # ← edit me
!python training/colab_smoke_test.py --hf-repo $HF_USER/tenacious-smoke-test

python3: can't open file '/content/tenacious-bench/training/colab_smoke_test.py': [Errno 2] No such file or directory


## Step 6 — Verify

If you see **`[smoke] ALL CHECKS PASSED`** above and the adapter is visible at `https://huggingface.co/<your-hf-user>/tenacious-smoke-test`, you are cleared to run `train.ipynb` on Day 5.
